[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/60_mla_solution.ipynb)

# 🔴 Solution: Multi-Head Latent Attention (MLA)

Reference solution for DeepSeek-style Multi-Head Latent Attention, which compresses KV into a low-rank latent vector to reduce KV cache size.

$$c_{KV} = X W_{dkv}, \quad K = c_{KV} W_{uk}, \quad V = c_{KV} W_{uv}, \quad Q = X W_q$$

$$\text{MLA}(X) = \text{MultiHeadAttn}(Q, K, V)$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def mla_attention(X, W_dkv, W_uk, W_uv, W_q, num_heads):
    B, S, D = X.shape
    D_h = W_q.shape[1] // num_heads
    c_kv = X @ W_dkv
    K = c_kv @ W_uk
    V = c_kv @ W_uv
    Q = X @ W_q
    def split_heads(t):
        return t.view(B, S, num_heads, D_h).transpose(1, 2)
    Q, K, V = split_heads(Q), split_heads(K), split_heads(V)
    scale = D_h ** -0.5
    attn = torch.softmax(Q @ K.transpose(-2, -1) * scale, dim=-1)
    out = (attn @ V).transpose(1, 2).reshape(B, S, num_heads * D_h)
    return out

In [ ]:
torch.manual_seed(0)
B, S, D = 2, 6, 32
num_heads = 4
D_h = 8          # head dim
D_c = 8          # compressed KV dim (latent)

W_dkv = torch.randn(D, D_c) * 0.1      # compress to latent
W_uk  = torch.randn(D_c, num_heads * D_h) * 0.1  # up-project to K
W_uv  = torch.randn(D_c, num_heads * D_h) * 0.1  # up-project to V
W_q   = torch.randn(D, num_heads * D_h) * 0.1

X = torch.randn(B, S, D)

# Show compression: c_kv is much smaller than full KV
c_kv = X @ W_dkv
K_full = c_kv @ W_uk
print(f"Input shape:          {X.shape}")       # (2, 6, 32)
print(f"Compressed KV shape:  {c_kv.shape}")    # (2, 6, 8)  <-- small latent
print(f"Full K shape:         {K_full.shape}")  # (2, 6, 32) <-- expanded

out = mla_attention(X, W_dkv, W_uk, W_uv, W_q, num_heads)
print(f"Output shape:         {out.shape}")     # (2, 6, 32)

In [ ]:
from torch_judge import check
check("mla")